In [34]:
import numpy as np
import scipy.stats as stats
import pandas as pd
import matplotlib.pyplot as plt

In [35]:
PUBLIC_SRC = 'https://seshat.datasd.org/get_it_done_reports'
YEARS = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
PRIVATE_CSV = '../data/gid_cases-001.csv'
ATTACH_CSV = '../data/public-attachments-cases.csv'
REFFERALS_CSV = '../data/referral data for UCSD-2026-11-02-23-09-16.csv'
WINDOW_START, WINDOW_END = '2016-01-01', '2026-10-01'

# Loading Datasets

## Loading public and private GID data

In [36]:
PUB_COLS = [
    'service_request_id', 'service_request_parent_id', 'date_requested', 'case_age_days',
    'case_record_type', 'service_name', 'service_name_detail', 'date_closed', 'status', 'case_origin', 'referred', 'public_description'
    ]

closed = pd.concat([pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_closed_{y}_datasd.csv', usecols=PUB_COLS) for y in YEARS], ignore_index=True)
open_ = pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_open_datasd.csv', usecols=PUB_COLS)

C:\Users\cjjen\AppData\Local\Temp\ipykernel_26700\1812712720.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  closed = pd.concat([pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_closed_{y}_datasd.csv', usecols=PUB_COLS) for y in YEARS], ignore_index=True)


In [37]:
pub = pd.concat([closed, open_], ignore_index=True).drop_duplicates('service_request_id')

pub['date_requested'] = pd.to_datetime(pub['date_requested'])
pub['date_requested'] = pub['date_requested'].dt.tz_localize('UTC').dt.tz_convert('US/Pacific')
pub['date_closed'] = pd.to_datetime(pub['date_closed'])

pub = pub[(pub['date_requested'] >= WINDOW_START) & (pub['date_requested'] < WINDOW_END)].copy()

In [38]:
PRIV_COLS = [
    'Case Reason', 'Last Modified Date', 'Parent Case ID', 'Parent Case ID', 'Case Number',
    'Owner ID', 'Last Modified By ID', 'Created By ID', 'Created Date',
    ]

RENAME = {t:t.lower().replace(' ', '_') for t in PRIV_COLS}
RENAME.update({'Status': 'private_status'})
priv = pd.read_csv(PRIVATE_CSV, usecols=list(RENAME), dtype=str).rename(columns=RENAME)

In [39]:
priv['case_number'] = pd.to_numeric(priv['case_number'], errors='coerce')
priv = priv.dropna(subset=['case_number'])
priv['case_number'] = priv['case_number'].astype(int)
priv.drop_duplicates('case_number', inplace=True)

priv['last_modified_date'] = pd.to_datetime(priv['last_modified_date'])
priv['owner_type'] = priv['owner_id'].str[:3].map({'005': 'worker', '00G': 'queue'}).fillna('other')
priv['modified_by_worker'] = priv['last_modified_by_id'].str[:3].eq('005')
priv['case_number'] = priv['case_number'].astype(float)
priv.drop(columns=['owner_id', 'last_modified_by_id'], inplace=True)

In [40]:
datasd = pub.merge(priv, left_on='service_request_id', right_on='case_number', how='left')

## Loading referrals data

In [41]:
REF_COLS = ['Internal Case Referral Flag', 'Internal Referral Count']

REF_RENAME = {t:t.lower().replace(' ', '_') for t in REF_COLS}
REF_RENAME.update({'Case Number': 'ref_case_number', 'Status': 'referral_status'})
referrals_info = pd.read_csv(REFFERALS_CSV, encoding='latin-1', usecols=list(REF_RENAME)).rename(columns=REF_RENAME)

In [48]:
df = datasd.merge(referrals_info, left_on='service_request_id', right_on='ref_case_number', how='left').drop(columns='ref_case_number')

In [49]:
df['last_modified_date'] = df['last_modified_date'].dt.tz_localize('US/Pacific', ambiguous=True)
df['date_closed'] = pd.to_datetime(df['date_closed']).dt.tz_localize('US/Pacific')

In [50]:
parents = df.loc[df['case_age_days'] < 0, 'service_request_parent_id'].dropna().unique()
df = df[~(df['service_request_parent_id'].isin(parents) | df['service_request_id'].isin(parents))]

In [52]:
df.fillna({'internal_case_referral_flag':0, 'internal_referral_count':0}, inplace=True)
df.drop(columns=['referral_status'], inplace=True)
df = df[(df['case_age_days'] >= 0) & (df['status'].isin(['Closed', 'Referred']))]

In [53]:
df.loc[df['internal_referral_count'] > 0, 'internal_case_referral_flag'] = 1.0

In [54]:
df.loc[(df['internal_case_referral_flag'] == 1.) & (df['internal_referral_count'] == 0.), ['service_request_id', 'date_requested', 'case_age_days', 'case_record_type', 'service_name', 'date_closed', 'status', 'referred', 'case_reason', 'private_status', 'internal_referral_count', 'internal_case_referral_flag']]

,service_request_id,date_requested,case_age_days,case_record_type,service_name,date_closed,status,referred,case_reason,private_status,internal_referral_count,internal_case_referral_flag
2333225,4780714,2024-06-03 15:19:00-07:00,1.0,TSW,Graffiti - Public,2024-06-04 00:00:00-07:00,Referred,This report has been referred to Caltrans San Diego at ct.public.information.d11@dot.ca.gov,Problem resolved,Closed - Referred,0.0,1.0


In [55]:
df.loc[df['service_request_id'] == 4780714, 'internal_case_referral_flag'] = 0.

In [56]:
# Inplace function(s) and code to clean up referred column. Based on the frequency of referrals made
def categorize_referrals(txt):
    txt = str(txt).lower()
    if 'sdge' in txt or 'sdg&e' in txt:
        return 'sdg&e'
    if 'cox.com' in txt:
        return 'cox communications'
    if 'charter.com' in txt or 'spectrum' in txt:
        return 'spectrum'
    if 'att' in txt or 'at&t' in txt:
        return 'at&t'
    if 'cleanharbors' in txt:
        return 'clean harbors sanitation'
    if 'wm.com' in txt or 'waste management' in txt:
        return 'waste management'
    if 'republicservices' in txt or 'republic-services' in txt:
        return 'republic services'
    if 'edcodisposal' in txt:
        return 'edco disposal'
    if 'sdmts' in txt:
        return 'mts'
    if 'nctd.org' in txt:
        return 'nctd'
    if 'bnsf' in txt:
        return 'bnsf railroad'
    if 'caltrans' in txt or 'dot.ca.gov' in txt:
        return 'caltrans'
    if 'portofsandiego' in txt or 'port of san diego' in txt:
        return 'port of sd'
    if 'san.org' in txt:
        return 'sd international airport'
    if 'sandi.net' in txt:
        return 'sd unified school district'
    if 'ucsd.edu' in txt:
        return 'ucsd'
    if 'sdsu.edu' in txt:
        return 'sdsu'
    if 'sdccd.edu' in txt:
        return 'sdccd'
    if 'chulavistaca.gov' in txt:
        return 'city of chula vista'
    if 'sdcounty.ca.gov' in txt:
        return 'san diego county'
    if 'parks.ca.gov' in txt or 'wildlife.ca.gov' in txt:
        return 'california state parks/wildlife'
    if 'usps' in txt:
        return 'usps'
    if 'livelmh' in txt:
        return 'liberty military housing'
    if 'improvedtsd' in txt or 'ballparkdistrict' in txt or 'downtownsandiego.org' in txt:
        return 'downtown clean & safe / ballpark'
    if 'sdhumane' in txt:
        return 'sd humane society'
    if 'vector' in txt:
        return 'county vector control'
    if 'rms' in txt or 'retailsolutions' in txt or 'retailms.net' in txt:
        return 'retail solutions (rms)'
    if 'smartcart' in txt:
        return 'smart cart'
    if 'clearchannel' in txt:
        return 'clear channel'
    if 'outfront.com' in txt:
        return 'outfront media'
    if 'engineering' in txt:
        return 'field engineering'
    if 'facilitiesmaint' in txt:
        return 'facilities maintenance'
    if 'treepermit' in txt:
        return 'tree permits'
    if 'publicliability' in txt or 'riskmanagement' in txt:
        return 'risk management / public liability'
    if 'specialevents' in txt:
        return 'special events'
    if 'realestate' in txt:
        return 'real estate assets'
    if 'sdpd' in txt or 'police' in txt:
        return 'sdpd'
    if 'sdfd' in txt or 'fire' in txt:
        return 'sdfd'
    if 'wcm_gid' in txt or 'water' in txt:
        return 'water / wastewater (pud)'
    if 'jkafulides' in txt or 'parking' in txt:
        return 'parking meter repair'
    if 'sdlg' in txt or 'lifeguard' in txt:
        return 'lifeguards'
    if 'fit_dsd' in txt or 'development services' in txt:
        return 'development services (dsd)'
    if 'sdbusiness' in txt:
        return 'sd business / economic dev'
    if 'street division' in txt:
        return 'street division'
    if 't-row code compliance' in txt:
        return 't-row code compliance'     
    if 'sanitization contractor' in txt or 'environmental services' in txt or 'esd' in txt:
        return 'environmental services (esd)'
    if 'parks intake' in txt:
        return 'parks & recreation'
    if 'testing only' in txt:
        return 'test/invalid'
    if 'null' in txt:
        return 'error'
    return txt

def clean_referrals(df):
    df['referred'] = df['referred'].apply(categorize_referrals)
    # If the referral is not a specific organization (is an email), extract the email:
    df.loc[df['referred'].str.contains('@'), 'referred'] = df.loc[df['referred'].str.contains('@'), 'referred'].str.extract(r'([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})')[0].str.lower()
    df.loc[df['referred'] == 'nan', 'referred'] = np.nan

In [57]:
clean_referrals(df)

In [58]:
df.columns

Index(['service_request_id', 'service_request_parent_id', 'date_requested',
       'case_age_days', 'case_record_type', 'service_name',
       'service_name_detail', 'date_closed', 'status', 'case_origin',
       'referred', 'public_description', 'case_number', 'created_by_id',
       'created_date', 'last_modified_date', 'parent_case_id', 'case_reason',
       'private_status', 'owner_type', 'modified_by_worker',
       'internal_case_referral_flag', 'internal_referral_count'],
      dtype='object')

In [60]:
# %% Split cases by internal referral count, then describe each case_record_type for each group
# Assumes `df` with: case_record_type, service_name, internal_referral_count, public_description
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

THRESHOLD = 3                 # internal_referral_count >= 3 -> "routed incorrectly" (working definition agreed with the City)
EXCLUDE_ENCAMPMENT = False     # Marcus: Encampment is routinely routed to ~3 departments, so it is not a misroute signal
MIN_N = 30                    # need at least this many descriptions to describe a group
USE_LLM = False               # True = also write short prose descriptions with Claude (see the last section)

d = df.copy()
d['internal_referral_count'] = pd.to_numeric(d['internal_referral_count'], errors='coerce')
print(f'internal_referral_count missing: {d["internal_referral_count"].isna().sum():,} rows (treated as 0 = no internal referrals)')
d['internal_referral_count'] = d['internal_referral_count'].fillna(0)

# ---------------------------------------------------------------
# 1. The two datasets
# ---------------------------------------------------------------
routed_correctly = d[d['internal_referral_count'] < THRESHOLD].copy()
routed_incorrectly = d[d['internal_referral_count'] >= THRESHOLD].copy()
if EXCLUDE_ENCAMPMENT:
    enc = routed_incorrectly['service_name'].eq('Encampment')   # verify: is "Encampment" stored in service_name in your data?
    print(f'Encampment cases with {THRESHOLD}+ referrals removed from "routed incorrectly": {enc.sum():,} (left in neither group)')
    routed_incorrectly = routed_incorrectly[~enc]

good_by_type = routed_correctly.groupby('case_record_type')       # the "grouped by case_record_type" views
bad_by_type = routed_incorrectly.groupby('case_record_type')

counts = pd.DataFrame({'routed_correctly': good_by_type.size(), 'routed_incorrectly': bad_by_type.size()}).fillna(0).astype(int)
counts['pct_incorrect'] = (counts['routed_incorrectly'] / (counts['routed_correctly'] + counts['routed_incorrectly']) * 100).round(2)
print(f'\nrouted correctly: {len(routed_correctly):,} rows | routed incorrectly: {len(routed_incorrectly):,} rows')
display(counts.sort_values('routed_incorrectly', ascending=False))

# ---------------------------------------------------------------
# 2. Condense the descriptions (no API needed): key terms, typical example, and what stands out in the "incorrect" group
# ---------------------------------------------------------------
def clean_text(frame):
    s = frame['public_description'].dropna().astype(str)
    return s[s.str.strip().str.len() > 0]

fit_docs = pd.concat([clean_text(routed_correctly), clean_text(routed_incorrectly)])
vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=5, max_features=50000, sublinear_tf=True,
                      token_pattern=r'(?u)\b[A-Za-z][A-Za-z]+\b')      # words only: ignore street numbers and phone fragments
vec.fit(fit_docs.sample(min(len(fit_docs), 300000), random_state=0))
terms = np.array(vec.get_feature_names_out())

def mean_vec(texts):
    return np.asarray(vec.transform(texts).mean(axis=0)).ravel()

def top_terms(texts, k=8):
    # highest-scoring terms, skipping near-duplicates (e.g. keep "illegal dumping", skip "dumping" and "illegal")
    chosen = []
    for t in terms[np.argsort(mean_vec(texts))[::-1][:200]]:
        w = set(t.split())
        if any(w <= set(c.split()) or set(c.split()) <= w for c in chosen):
            continue
        chosen.append(t)
        if len(chosen) == k:
            break
    return ', '.join(chosen)

def typical_example(texts, width=160):
    X = vec.transform(texts)
    sims = np.asarray(X.dot(np.asarray(X.mean(axis=0)).T)).ravel()          # closest to the group's average
    best = texts.iloc[int(np.argmax(sims))].replace('\n', ' ').strip()
    return best if len(best) <= width else best[:width].rstrip() + '...'

rows = []
for rt in sorted(set(routed_correctly['case_record_type'].dropna()) | set(routed_incorrectly['case_record_type'].dropna())):
    row = {'case_record_type': rt}
    samples = {}
    for label, frame in (('routed_correctly', routed_correctly), ('routed_incorrectly', routed_incorrectly)):
        texts = clean_text(frame[frame['case_record_type'] == rt])
        row[f'n_{label}'] = int((frame['case_record_type'] == rt).sum())
        row[f'n_desc_{label}'] = len(texts)
        if len(texts) < MIN_N:
            row[f'{label}_description'] = f'too few descriptions (n={len(texts)})'
            continue
        s = texts.sample(min(len(texts), 20000), random_state=0)
        samples[label] = s
        row[f'{label}_key_terms'] = top_terms(s)
        row[f'{label}_example'] = typical_example(s)
        row[f'{label}_description'] = f'Mostly about: {row[f"{label}_key_terms"]}. Typical: "{row[f"{label}_example"]}"'
    if len(samples) == 2:      # words that are more common in the incorrect group than in the correct group
        diff = mean_vec(samples['routed_incorrectly']) - mean_vec(samples['routed_correctly'])
        row['more_common_in_incorrect'] = ', '.join(terms[np.argsort(diff)[::-1][:8]])
    rows.append(row)

desc_table = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 200)
display(desc_table[['case_record_type', 'n_routed_correctly', 'n_routed_incorrectly',
                    'routed_correctly_description', 'routed_incorrectly_description']])
# desc_table.to_csv('record_type_descriptions.csv', index=False)     # small file; do not commit the full datasets (GitHub size limits)

# ---------------------------------------------------------------
# 3. Optional: have Claude write 2-sentence descriptions from a sample (needs an API key; costs a few cents)
# ---------------------------------------------------------------
# PRIVACY: descriptions are resident-written and can contain names, phone numbers, or addresses. Check with the City before
# sending them to any outside service. The scrub below removes emails and phone numbers only, not names or addresses.
EMAIL = r'[\w.+-]+@[\w-]+\.[\w.]+'
PHONE = r'\(?\b\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b'
def scrub(s):
    return re.sub(PHONE, '[phone]', re.sub(EMAIL, '[email]', s))

def llm_description(texts, record_type, label, n_sample=100, client=None, model='claude-haiku-5-5'):
    sample = texts.sample(min(len(texts), n_sample), random_state=0).map(scrub).str.slice(0, 300).tolist()
    quality = 'were routed with fewer than 3 internal referrals' if label == 'routed_correctly' else 'had 3 or more internal referrals'
    prompt = (f"Below are resident-written descriptions of reports filed under the City of San Diego case type '{record_type}'. "
              f"These reports {quality}. In at most 2 sentences, describe what this case type is about and what the reports "
              "typically say. Use only what appears in the descriptions; do not guess.\n\n" + '\n'.join(f'- {t}' for t in sample))
    resp = client.messages.create(model=model, max_tokens=150, messages=[{'role': 'user', 'content': prompt}])
    return resp.content[0].text.strip()

if USE_LLM:
    import anthropic
    client = anthropic.Anthropic()          # reads the ANTHROPIC_API_KEY environment variable
    for i, row in desc_table.iterrows():
        for label, frame in (('routed_correctly', routed_correctly), ('routed_incorrectly', routed_incorrectly)):
            texts = clean_text(frame[frame['case_record_type'] == row['case_record_type']])
            if len(texts) >= MIN_N:
                desc_table.loc[i, f'{label}_llm_description'] = llm_description(texts, row['case_record_type'], label, client=client)
    display(desc_table[['case_record_type', 'routed_correctly_llm_description', 'routed_incorrectly_llm_description']])

internal_referral_count missing: 0 rows (treated as 0 = no internal referrals)

routed correctly: 3,082,594 rows | routed incorrectly: 3,604 rows


,routed_correctly,routed_incorrectly,pct_incorrect
case_record_type,,,
TSW,1057864,958,0.09
ESD Complaint/Report,1042403,679,0.07
Parking,515664,482,0.09
Parks & Recreation,50280,396,0.78
DSD,20633,338,1.61
TSW ROW,49415,313,0.63
Neighborhood Policing,286471,215,0.07
Traffic Engineering,29183,160,0.55
Storm Water Code Enforcement,27900,59,0.21


,case_record_type,n_routed_correctly,n_routed_incorrectly,routed_correctly_description,routed_incorrectly_description
0,DSD,20633,338,"Mostly about: airbnb, noise, property, com, construction, illegal, rooms, stro. Typical: ""3551 FRONT ST. STR-08221L, STR-08216L, STR-08215L, STR-03515L. Missing required signage / missing required...","Mostly about: sidewalk, illegal, property, graffiti, street, city, sign, hazard. Typical: ""Illegal sign posted on public property."""
1,ESD Abatement,397,1,"Mostly about: homeless, encampment, camp, trash, tent, camping, illegal, sidewalk. Typical: ""Homeless encampment""",too few descriptions (n=1)
2,ESD Complaint/Report,1042403,679,"Mostly about: trash, missed, picked, dumping, recycling, collection, illegal, street. Typical: ""Missed trash can""","Mostly about: trash, dumping, illegal, sidewalk, street, homeless, encampment, abandoned. Typical: ""Trash"""
3,ESD Event/Service,1,0,too few descriptions (n=1),too few descriptions (n=0)
4,Homeless Outreach,1402,3,"Mostly about: encampment, homeless, camp, sidewalk, living, camping, parking, parked. Typical: ""Homeless Encampment""",too few descriptions (n=3)
5,Neighborhood Policing,286471,215,"Mostly about: encampment, homeless, encampments, camp, sidewalk, camping, living, people. Typical: ""Homeless encampment""","Mostly about: encampment, homeless, parking, living, rv, camping, parked, people. Typical: ""Homeless encampment"""
6,PUD Code Compliance,8,0,too few descriptions (n=8),too few descriptions (n=0)
7,Parking,515664,482,"Mostly about: parked, car, parking, vehicle, abandoned, weeks, moved, days. Typical: ""Car 72 overhours parked""","Mostly about: abandoned, parking, parked, trailer, car, vehicle, boat, street. Typical: ""There's an abandoned boat on a trailer and another trailer that has been parked there for over 3 weeks."""
8,Parks & Recreation,50280,396,"Mostly about: graffiti, park, broken, water, trash, beach, dog, playground. Typical: ""Graffiti at lindberg park""","Mostly about: graffiti, park, trash, encampment, dumping, illegal, canyon, tree. Typical: ""Graffiti"""
9,Special Situation,973,0,"Mostly about: masks, dining, covid, people, party, distancing, social, indoor. Typical: ""Indoor and outdoor dining. No masks. No social distancing""",too few descriptions (n=0)


In [61]:
desc_table.drop(columns=['routed_correctly_example', 'routed_incorrectly_example'])

,case_record_type,n_routed_correctly,n_desc_routed_correctly,routed_correctly_key_terms,routed_correctly_description,n_routed_incorrectly,n_desc_routed_incorrectly,routed_incorrectly_key_terms,routed_incorrectly_description,more_common_in_incorrect
0,DSD,20633,20234,"airbnb, noise, property, com, construction, illegal, rooms, stro","Mostly about: airbnb, noise, property, com, construction, illegal, rooms, stro. Typical: ""3551 FRONT ST. STR-08221L, STR-08216L, STR-08215L, STR-03515L. Missing required signage / missing required...",338,338,"sidewalk, illegal, property, graffiti, street, city, sign, hazard","Mostly about: sidewalk, illegal, property, graffiti, street, city, sign, hazard. Typical: ""Illegal sign posted on public property.""","sidewalk, graffiti, illegal sign, illegal, tree, hazard, signs, sign"
1,ESD Abatement,397,356,"homeless, encampment, camp, trash, tent, camping, illegal, sidewalk","Mostly about: homeless, encampment, camp, trash, tent, camping, illegal, sidewalk. Typical: ""Homeless encampment""",1,1,NaN,too few descriptions (n=1),NaN
2,ESD Complaint/Report,1042403,764959,"trash, missed, picked, dumping, recycling, collection, illegal, street","Mostly about: trash, missed, picked, dumping, recycling, collection, illegal, street. Typical: ""Missed trash can""",679,677,"trash, dumping, illegal, sidewalk, street, homeless, encampment, abandoned","Mostly about: trash, dumping, illegal, sidewalk, street, homeless, encampment, abandoned. Typical: ""Trash""","sidewalk, homeless, boat, canyon, graffiti, trash, illegal, litter"
3,ESD Event/Service,1,1,NaN,too few descriptions (n=1),0,0,NaN,too few descriptions (n=0),NaN
4,Homeless Outreach,1402,1402,"encampment, homeless, camp, sidewalk, living, camping, parking, parked","Mostly about: encampment, homeless, camp, sidewalk, living, camping, parking, parked. Typical: ""Homeless Encampment""",3,3,NaN,too few descriptions (n=3),NaN
5,Neighborhood Policing,286471,281074,"encampment, homeless, encampments, camp, sidewalk, camping, living, people","Mostly about: encampment, homeless, encampments, camp, sidewalk, camping, living, people. Typical: ""Homeless encampment""",215,214,"encampment, homeless, parking, living, rv, camping, parked, people","Mostly about: encampment, homeless, parking, living, rv, camping, parked, people. Typical: ""Homeless encampment""","parking, rv, parked, living, dumping, illegal, rv parked, camping"
6,PUD Code Compliance,8,8,NaN,too few descriptions (n=8),0,0,NaN,too few descriptions (n=0),NaN
7,Parking,515664,502349,"parked, car, parking, vehicle, abandoned, weeks, moved, days","Mostly about: parked, car, parking, vehicle, abandoned, weeks, moved, days. Typical: ""Car 72 overhours parked""",482,482,"abandoned, parking, parked, trailer, car, vehicle, boat, street","Mostly about: abandoned, parking, parked, trailer, car, vehicle, boat, street. Typical: ""There's an abandoned boat on a trailer and another trailer that has been parked there for over 3 weeks.""","trailer, boat, dumping, illegal, sidewalk, vehicles, abandoned boat, illegal dumping"
8,Parks & Recreation,50280,44389,"graffiti, park, broken, water, trash, beach, dog, playground","Mostly about: graffiti, park, broken, water, trash, beach, dog, playground. Typical: ""Graffiti at lindberg park""",396,396,"graffiti, park, trash, encampment, dumping, illegal, canyon, tree","Mostly about: graffiti, park, trash, encampment, dumping, illegal, canyon, tree. Typical: ""Graffiti""","encampment, dumping, trash, seawall, illegal dumping, illegal, sidewalk, canyon"
9,Special Situation,973,905,"masks, dining, covid, people, party, distancing, social, indoor","Mostly about: masks, dining, covid, people, party, distancing, social, indoor. Typical: ""Indoor and outdoor dining. No masks. No social distancing""",0,0,NaN,too few descriptions (n=0),NaN


In [ ]:
desc_table.to_csv('desc_table.csv', index=False)

In [68]:
miss = pd.read_csv('../data/mishandled_cases_targets.csv')

In [70]:
random_sample = miss.groupby('case_record_type').sample(n=1, random_state=42)

random_sample

,service_request_id,case_record_type
76,4985524,DSD
146,5710970,ESD Complaint/Report
148,5817080,Neighborhood Policing
13,4764264,Parking
77,5076193,Parks & Recreation
141,5457929,TSW
83,5088860,TSW ROW


In [65]:
mishandled = pd.read_csv('../data/mishandled_cases_data.csv')

In [71]:
right = pd.read_csv('../data/resolved_cases_targets-2.csv')

In [72]:
random_sample_right = right.groupby('case_record_type').sample(n=1, random_state=42)

random_sample_right

,service_request_id,case_record_type
10047,5085995,DSD
3944,4941377,ESD Complaint/Report
259,4800064,Neighborhood Policing
14438,6104110,Parking
5372,4966063,Parks & Recreation
4973,5035475,Storm Water Code Enforcement
9999,5099008,TSW
11615,5253871,TSW ROW
12879,5559777,Traffic Engineering


In [66]:
resolved = pd.read_csv('../data/resolved_cases_data-2.csv')

In [76]:
mishandled[mishandled['service_request_id'].isin(random_sample['service_request_id'])]

,service_request_id,date_requested,service_name,service_name_detail,lat,lng,street_address,zipcode,council_district,comm_plan_code,comm_plan_name,park_name,case_origin,iamfloc,public_description
13,4764264,2024-05-22 11:48:00-07:00,Parking Violation,72 Hour Violation,32.804340,-117.177941,"3349-3509 Acworth Ave, San Diego, CA 92111, USA",92111.0,2.0,6.0,CLAIREMONT MESA,NaN,Web,SS-001700,Abandoned trailer full of junk and no plates or tags
76,4985524,2024-10-28 10:47:00-07:00,Other,NaN,32.738168,-117.160458,3138 5th Ave,92103.0,3.0,42.0,UPTOWN,NaN,Mobile,SS-000159-PV1,This fin sign goes up in the public right away every day on fifth Avenue between Redwood and Spruce. I have not received the answer I requested about what the California pedal code or city municip...
77,5076193,2025-01-16 11:56:00-08:00,Parks Issue,Graffiti,32.747661,-117.100471,4367 Wightman St,92105.0,9.0,56.0,MID-CITY:CITY HEIGHTS,NaN,Mobile,NaN,BACT
83,5088860,2025-01-27 13:20:00-08:00,Graffiti - Code Enforcement,Utility Box,32.747974,-117.101855,3801 43rd St,92105.0,9.0,56.0,MID-CITY:CITY HEIGHTS,NaN,Mobile,SS-001046,JAVI 43VR
141,5457929,2025-11-19 01:59:00-08:00,ROW Maintenance,TRAFFIC MAINTENANCE - OTHER,32.736738,-117.161351,3002 Fourth Ave,92103.0,3.0,42.0,UPTOWN,NaN,Mobile,SS-000116,"Quince & 4th Bus Stop is frequently mess. City Storm drain contractor placed 1/2 doz. sandbags here over a year ago, 1 on handicap ramp is soil for invasive grass blocking ramp. They moved broken ..."
145,5710970,2026-05-08 14:26:00-07:00,Other,NaN,32.756243,-117.121386,4346 Felton St,92104.0,9.0,59.0,MID-CITY:NORMAL HEIGHTS,NaN,Mobile,SS-010993,My case was closed without any explanation in the app. What do I do? I have no trashcan. And I have no recycling. Would you like to not Bill me for these two Services that I cannot use? I don?t ne...
146,5710970,2026-05-08 14:26:00-07:00,Other,NaN,32.756243,-117.121386,4346 Felton St,92104.0,9.0,59.0,MID-CITY:NORMAL HEIGHTS,NaN,Mobile,SS-010993,My case was closed without any explanation in the app. What do I do? I have no trashcan. And I have no recycling. Would you like to not Bill me for these two Services that I cannot use? I don?t ne...
148,5817080,2026-07-20 22:15:00-07:00,Encampment,NaN,32.893601,-117.151969,7765 Arjons Drive,92126.0,6.0,15.0,MIRA MESA,NaN,Mobile,SS-002732,beyond 3 day limit


In [78]:
df[df['service_request_id'] == 4985524]

,service_request_id,service_request_parent_id,date_requested,case_age_days,case_record_type,service_name,service_name_detail,date_closed,status,case_origin,...,created_by_id,created_date,last_modified_date,parent_case_id,case_reason,private_status,owner_type,modified_by_worker,internal_case_referral_flag,internal_referral_count
2643694,4985524,4798503.0,2024-10-28 10:47:00-07:00,169.0,DSD,Other,NaN,2025-04-15 00:00:00-07:00,Closed,Mobile,...,005t0000000F4uMAAS,2024-10-28 17:47:38.000,2025-04-15 16:16:58-07:00,5008z0000085bl9AAA,Closed-Duplicate Case,Closed,worker,True,1.0,1.0
